# Data Preprocessing and Random Forest Training

## Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import json

# Pipeline and Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Modeling and Evaluation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)
print("Libraries imported successfully.")

Libraries imported successfully.


## Data Loading & Initial Cleanup

In [2]:
DATA_PATH = "../data/training/training_data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset shape: {df.shape}")

# Drop irrelevant columns
cols_to_drop = ['Resume_ID', 'Name', 'Salary Expectation ($)']
existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)
print(f"Successfully dropped the following columns: {existing_cols}")

# Define Target and Features based on continuous 0-100 score
TARGET_COL = 'AI Score (0-100)'

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Split the data first to prevent data leakage during preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Loaded dataset shape: (1000, 17)
Successfully dropped the following columns: ['Resume_ID', 'Name', 'Salary Expectation ($)']
Training features shape: (800, 13)
Testing features shape: (200, 13)


## Feature Engineering Pipeline

In [11]:
# Custom Tokenizer
def split_comma_skills(skill_string):
    if not isinstance(skill_string, str):
        return []
    return [skill.strip() for skill in skill_string.split(',')]

# Feature Groups
num_features = [
    'Experience (Years)', 
    'Projects Count',
    'Structural Adherence', 
    'Adaptive Fluidity', 
    'Interpersonal Influence', 
    'Execution Velocity', 
    'Psychological Resilience'
]

cat_features = [
    'Education', 
    'Certifications', 
    'Job Role', 
    'Job Hopping'
]

text_feature = ['Skills'] 

# Numeric Transformer: Fill missing with 0
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

# Categorical Transformer: Fill missing with 'None' and One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Text Transformer for Skills: Fill missing, flatten, and vectorize
text_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', FunctionTransformer(np.ravel)), 
    ('tfidf', TfidfVectorizer(
        max_features=100, 
        tokenizer=split_comma_skills, 
        token_pattern=None 
    ))
])

# Combine all transformers into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features),
        ('txt', text_transformer, text_feature)
    ],
    remainder='drop'
)

print("Preprocessing pipeline constructed successfully.")

Preprocessing pipeline constructed successfully.


## Model Pipeline Assembly & Training

In [4]:
# Initialize the Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1 # Uses all available CPU cores
)

# Create the unified modeling pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', rf_model)
])

# Train the model
print("Training the Random Forest pipeline...")
pipeline.fit(X_train, y_train)
print("Training complete!")

Training the Random Forest pipeline...
Training complete!


## Model Evaluation

In [5]:
y_pred = pipeline.predict(X_test) # Generate predictions on the test set

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Model Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} points")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} points")
print(f"R-squared (R2) Score: {r2:.4f}")

--- Model Evaluation ---
Mean Absolute Error (MAE): 2.86 points
Root Mean Squared Error (RMSE): 4.40 points
R-squared (R2) Score: 0.9638


## Extracting Feature Names for XAI

In [6]:
def get_feature_names(column_transformer):
    """Extracts feature names from a fitted ColumnTransformer."""
    output_features = []

    for name, pipe, features in column_transformer.transformers_:
        if name == 'drop':
            continue

        if name == 'num':
            output_features.extend(features)

        elif name == 'cat':
            # Extract names from the OneHotEncoder inside the pipeline
            ohe = pipe.named_steps['onehot']
            cat_features = ohe.get_feature_names_out(features)
            output_features.extend(cat_features)

        elif name == 'txt':
            # Extract names from the TfidfVectorizer inside the pipeline
            tfidf = pipe.named_steps['tfidf']
            txt_features = tfidf.get_feature_names_out()
            # Prefix text features for clarity
            txt_features = [f"Skill_{feat}" for feat in txt_features]
            output_features.extend(txt_features)

    return output_features

# Extract and save the feature names
fitted_preprocessor = pipeline.named_steps['preprocessor']
feature_names = get_feature_names(fitted_preprocessor)

print(f"Total features extracted for TreeSHAP: {len(feature_names)}")
print("Sample of generated features:")
print(feature_names[:10]) # Print first 10 features to verify

Total features extracted for TreeSHAP: 37
Sample of generated features:
['Experience (Years)', 'Projects Count', 'Structural Adherence', 'Adaptive Fluidity', 'Interpersonal Influence', 'Execution Velocity', 'Psychological Resilience', 'Education_B.Sc', 'Education_B.Tech', 'Education_M.Tech']


## Saving the Model and Feature Names

In [7]:
# Save the trained pipeline
MODEL_PATH = "../api/saved_models/rf_model.pkl"
joblib.dump(pipeline, MODEL_PATH)
print(f"Model successfully saved to: {MODEL_PATH}")

# Save the feature names for TreeSHAP analysis
FEATURES_PATH = "../data/training/feature_names.json"
with open(FEATURES_PATH, "w") as f:
    json.dump(feature_names, f)
print(f"Feature names successfully saved to: {FEATURES_PATH}")

Model successfully saved to: ../api/saved_models/rf_model.pkl
Feature names successfully saved to: ../data/training/feature_names.json
